In [ ]:
# 형상 DOE for
    # 응력해석  
    # 온도/해석종류 
    # 무부하해석
# 전류/위상각 for
    # 해석
    # 데이터 추출
 # 다른 해석 FEA 추출   
    # 열해석


# 1) Environment / Imports (for debugging)

In [2]:
# 1) Environment / Imports (for debugging)
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np


# Ensure repo root (folder containing 'tools/' or legacy 'tool/') is on sys.path
repo_root = pathlib.Path.cwd().resolve()
while not ((repo_root / "tools").exists() or (repo_root / "tool").exists()) and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print("repo_root:", repo_root)

# Force reload of local package during iterative edits


# Import utilities extracted from the notebook
from tools.motorCAD.pyMCAD import (
    get_magnetic_data,
    get_magnetic_data_from_file,
    get_magnetic_timeseries_from_file,
    interactive_magnetic_plot,
    interactive_magnetic_quiver,
    interactive_b_locus_field_plot,
    get_element_loss_fields,
    interactive_loss_fields_plot,
    mcad_default_export_dir,
    mcad_make_temp_txt_path,
    find_latest_mes,
    )
import importlib
import tools.motorCAD.pyMCAD as _pyMCAD
importlib.reload(_pyMCAD)
# 1.5) Enable zoomable Matplotlib backend (ipympl)
try:
    import ipympl  # noqa: F401
    get_ipython().run_line_magic("matplotlib", "widget")
    import matplotlib
    print("matplotlib backend:", matplotlib.get_backend())
except Exception as e:
    print("ipympl/widget backend not available; using default backend. Error:", e)

import os
import matplotlib.pyplot as plt
import ansys.motorcad.core as pymotorcad
mc = pymotorcad.MotorCAD(open_new_instance=False)
# mc.initialise_tab_names()    # 꼭 필요한지

repo_root: D:\KangDH\Emlab_emach
matplotlib backend: widget


In [3]:
# Find *.mot files (uses repo-local pyAEDT utilities)
from tools.pyutils.import_path import ensure_repo_root_on_path

_ = ensure_repo_root_on_path()
from pyAEDT.aedt_file_utils import find_files

DOEDir = r"E:\\KDH\\AdaptiveTemplate\\TestCAD1\\optiSLangExport\\ExportedProj.opd\\Sensitivity__FromPara"
fileList = find_files(DOEDir, ".mot")

# print(f"Found {len(fileList)} .mot files")
# fileList[:5]  # preview

🔍 파일 검색 중...
📂 경로: E:\\KDH\\AdaptiveTemplate\\TestCAD1\\optiSLangExport\\ExportedProj.opd\\Sensitivity__FromPara
🔎 패턴: .mot
🔄 재귀 검색: 예 (무제한)
🔤 대소문자 구분: 아니오

✅ 총 4개의 파일 발견

📋 발견된 파일 목록:

  1. ExportedProj.mot
     경로: E:\KDH\AdaptiveTemplate\TestCAD1\optiSLangExport\ExportedProj.opd\Sensitivity__FromPara\Design0017\ExportedProj
     크기: 0.68 MB
     수정: 2026-01-19 13:05:44

  2. ExportedProj.mot
     경로: E:\KDH\AdaptiveTemplate\TestCAD1\optiSLangExport\ExportedProj.opd\Sensitivity__FromPara\Design0018\ExportedProj
     크기: 0.68 MB
     수정: 2026-01-19 13:05:43

  3. ExportedProj.mot
     경로: E:\KDH\AdaptiveTemplate\TestCAD1\optiSLangExport\ExportedProj.opd\Sensitivity__FromPara\Design0019\ExportedProj
     크기: 0.68 MB
     수정: 2026-01-21 10:04:45

  4. ExportedProj.mot
     경로: E:\KDH\AdaptiveTemplate\TestCAD1\optiSLangExport\ExportedProj.opd\Sensitivity__FromPara\Design0020\ExportedProj
     크기: 0.68 MB
     수정: 2026-01-19 13:12:22


# Parametric Sweep 설정

In [4]:
# Build sweep points and Motor-CAD case dictionaries
from tools.pyutils.import_path import ensure_repo_root_on_path

_ = ensure_repo_root_on_path()
from tools.pyutils.sweep import mkIpkPhaseMap, to_mcad_cases

ipeak_steps = 6
phase_steps = 8

ipeaks, phases, sweep_points = mkIpkPhaseMap(ipeak_steps, phase_steps)
cases = to_mcad_cases(sweep_points, ipeak_key="PeakCurrent", phase_key="PhaseAdvance")


In [12]:
# 2) Export + Parse (time series)
from pathlib import Path
from datetime import datetime

# Output file base name
out_dir = mcad_default_export_dir(mc)
base_filename = Path(out_dir) / "MagTransient.txt"
print("Output dir:", out_dir)
print("Base output file:", base_filename)

# Export settings
DO_EXPORT = True
first_step = 1
final_step = 45

def _unique_filename_if_exists(path: Path) -> Path:
    """If 'path' exists, return a new path with a timestamp suffix."""
    if not path.exists():
        return path
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return path.with_name(f"{path.stem}_{stamp}{path.suffix}")


Output dir: E:\KDH\AdaptiveTemplate
Base output file: E:\KDH\AdaptiveTemplate\MagTransient.txt


In [10]:
# List available .mes results by type (for the CURRENTLY OPEN .mot)
# NOTE: Outputs are saved under export dir you pass to process_fea_result_from_mes,
# or default is the folder containing the open .mot file (mcad_default_export_dir).
from tools.motorCAD.pyMCAD import list_mes_files

mes_by_kind = list_mes_files(mc)
for kind, paths in mes_by_kind.items():
    if not paths:
        continue
    print(f"{kind}: {len(paths)}")
    for p in paths[:5]:
        print("  ", p)

# export

In [16]:
# Batch export (CURRENTLY OPEN .mot 기준): FEResultsData의 모든 .mes를 종류별로 처리해서 GIF/SVG로 저장
# - OnLoadTorque_result_* : transient => Mag_b.gif, Mag_a.gif, Mag_j.gif
# - StaticLoad/StaticOC : Mag_b.svg, Mag_a.svg, Mag_j.svg
# - OnLoadLoss : Loss_Pt.svg, Loss_Phys.svg, Loss_Pj.svg, Loss_Peddy.svg (등)
# - Thermal : Thermal_t.svg, Thermal_g.svg, Thermal_q.svg
# - Centrifugal(stress) : Stress_svm.svg, Stress_sp1.svg, Stress_sp2.svg, Stress_sx.svg, Stress_sy.svg, Stress_txy.svg
# - Cogging : Cogging_*.svg
#
# 추가: OnLoadTorque transient는 txt를 이미 파싱하므로, 원하면 바로 .h5로도 저장 가능
from pathlib import Path
import importlib

PLOT_MODE = "interactive"   # "gif" => OnLoadTorque는 GIF, snapshot들은 SVG | "none" => plot 저장은 안 하고 export만 수행
GIF_FPS = 6
EXPORT_SUBDIR = "postproc"  # FEResultsData 아래에 저장할 폴더

# Transient export step range (OnLoadTorque일 때만 의미 있음)
FIRST_STEP = 1
FINAL_STEP = 45

# Parsed transient(ts) -> HDF5 저장 여부 (h5py 필요)
EXPORT_MAG_H5 = True

# H5 mesh coords 저장 방식:
# - "by_step": 모든 node를 step별로 저장 (정확하지만 용량 큼)
# - "by_step_moving_nodes": rotor + a2/a3/a4 같은 회전 영역 node만 step별 저장 (용량 절감)
MAG_H5_MESH_COORDS = "by_step_moving_nodes"

# 이미 mot가 열려있다고 가정 (테스트용). 필요하면 아래를 직접 해제해서 사용.
# mc.load_from_file(r"...")
mc.display_screen(r"E-Magnetics;FEA")

# Reload local helpers (tools/ 수정사항 즉시 반영)
import tools.motorCAD.pyMCAD.magnetic as _magnetic
import tools.motorCAD.pyMCAD.fea_workflow as _fea_workflow
import tools.motorCAD.pyMCAD.results as _mcad_results
importlib.reload(_magnetic)
importlib.reload(_mcad_results)
importlib.reload(_fea_workflow)

from tools.motorCAD.pyMCAD import list_mes_files
from tools.motorCAD.pyMCAD.fea_workflow import process_fea_result_from_mes

mes_by_kind = list_mes_files(mc)
total = sum(len(v) for v in mes_by_kind.values())
print(f"Found {total} .mes for current MOT")


Found 16 .mes for current MOT


In [17]:
mes_by_kind

{'Centrifugal': [WindowsPath('E:/KDH/AdaptiveTemplate/TestCAD1/FEResultsData/Centrifugal_result_.mes')],
 'Cogging': [WindowsPath('E:/KDH/AdaptiveTemplate/TestCAD1/FEResultsData/Cogging_result_1.mes')],
 'OnLoadLoss': [WindowsPath('E:/KDH/AdaptiveTemplate/TestCAD1/FEResultsData/OnLoadLoss_result_1.mes'),
  WindowsPath('E:/KDH/AdaptiveTemplate/TestCAD1/FEResultsData/OnLoadLoss_result_3.mes'),
  WindowsPath('E:/KDH/AdaptiveTemplate/TestCAD1/FEResultsData/OnLoadLoss_result_2.mes')],
 'OnLoadTorque': [WindowsPath('E:/KDH/AdaptiveTemplate/TestCAD1/FEResultsData/OnLoadTorque_result_1.mes'),
  WindowsPath('E:/KDH/AdaptiveTemplate/TestCAD1/FEResultsData/OnLoadTorque_result_3.mes'),
  WindowsPath('E:/KDH/AdaptiveTemplate/TestCAD1/FEResultsData/OnLoadTorque_result_2.mes')],
 'Other': [WindowsPath('E:/KDH/AdaptiveTemplate/TestCAD1/FEResultsData/OpenCircuitLoss_result_1.mes'),
  WindowsPath('E:/KDH/AdaptiveTemplate/TestCAD1/FEResultsData/OpenCircuitTransient_result_1.mes')],
 'StaticLoad': [Window

In [19]:
paths

[]

In [ ]:

for kind, paths in mes_by_kind.items():
    if not paths:
        continue

    print(f"\n--- {kind}: {len(paths)}")
    for mes_path in paths:
        mes_path = Path(mes_path)
        out_dir = mes_path.parent / EXPORT_SUBDIR
        out_dir.mkdir(parents=True, exist_ok=True)

        result = process_fea_result_from_mes(
            mc,
            mes_path=mes_path,
            plot_mode=PLOT_MODE,
            out_dir=out_dir,
            first_step=int(FIRST_STEP),
            final_step=int(FINAL_STEP),
            point_size=2,
            gif_fps=int(GIF_FPS),
            export_magnetic_h5=bool(EXPORT_MAG_H5),
            mag_h5_mesh_coords=str(MAG_H5_MESH_COORDS),
        )
        print("mes:", mes_path.name)
        print("  saved to:", out_dir)
        if getattr(result, "mag_h5_path", None):
            print("  mag_h5_path:", result.mag_h5_path)
        if getattr(result, "mag_gif_paths", None):
            print("  mag_gif_paths:", result.mag_gif_paths)
        if getattr(result, "mag_svg_paths", None):
            print("  mag_svg_paths:", result.mag_svg_paths)
        if getattr(result, "loss_svg_paths", None):
            print("  loss_svg_paths:", result.loss_svg_paths)
        if getattr(result, "thermal_svg_paths", None):
            print("  thermal_svg_paths:", result.thermal_svg_paths)
        if getattr(result, "stress_svg_paths", None):
            print("  stress_svg_paths:", result.stress_svg_paths)
        if getattr(result, "cogging_svg_path", None):
            print("  cogging_svg_path:", result.cogging_svg_path)



--- Centrifugal: 1


Output()

mes: Centrifugal_result_.mes
  saved to: E:\KDH\AdaptiveTemplate\TestCAD1\FEResultsData\postproc

--- Cogging: 1


MotorCADError: No points exist

# 2 .(From exported Mag_*.h5) Make GIF (timeseries) + SVG (static snapshot or selected step)


In [14]:
# (From by_step_moving_nodes H5) Interactive plots
#
# This cell finds an H5 exported with mesh_coords_mode='by_step_moving_nodes',
# loads it lazily as a time series, then calls the existing interactive plot helpers.

import importlib
from pathlib import Path

import tools.motorCAD.pyMCAD as _pyMCAD
import tools.motorCAD.pyMCAD.magnetic as _magnetic
importlib.reload(_magnetic)
importlib.reload(_pyMCAD)

from tools.motorCAD.pyMCAD import (
    get_magnetic_timeseries_from_file,
    interactive_magnetic_plot,
    interactive_magnetic_quiver,
    list_mes_files,
    diagnose_magnetic_h5_mesh_motion,
    inspect_magnetic_timeseries_h5,
    interactive_b_locus_field_plot,
 )

# Reuse globals from Cell 10 if present
EXPORT_SUBDIR_LOCAL = str(globals().get("EXPORT_SUBDIR", "postproc"))
POINT_SIZE_LOCAL = float(globals().get("POINT_SIZE", 2))
CMAP_LOCAL = str(globals().get("CMAP", "jet"))
QUANTITY_LOCAL = str(globals().get("GIF_QUANTITY", "b")).lower().strip()

# 1) Collect Mag_*.h5 candidates under each results folder's postproc directory
h5_files: list[Path] = []
mes_by_kind = list_mes_files(mc)
for kind, paths in mes_by_kind.items():
    for mes_path in paths:
        mes_path = Path(mes_path)
        cand_dir = mes_path.parent / EXPORT_SUBDIR_LOCAL
        if cand_dir.exists():
            h5_files.extend(sorted(cand_dir.glob("Mag_*.h5")))
h5_files = sorted(set(h5_files))
print("Found Mag_*.h5:", len(h5_files))

if not h5_files:
    raise FileNotFoundError("No Mag_*.h5 found. Run the export cell with EXPORT_MAG_H5=True first.")

# 2) Prefer the most recently modified by_step_moving_nodes timeseries H5
candidates: list[Path] = []
for p in h5_files:
    try:
        info = inspect_magnetic_timeseries_h5(p)
        fmt = str(info.get("attrs", {}).get("format", ""))
        mcm = str(info.get("attrs", {}).get("mesh_coords_mode", ""))
        if ("timeseries" in fmt.lower()) and (mcm.lower() == "by_step_moving_nodes"):
            candidates.append(p)
    except Exception:
        continue

if candidates:
    h5_path = max(candidates, key=lambda pp: pp.stat().st_mtime)
else:
    # fallback: most recent timeseries H5
    ts_cands: list[Path] = []
    for p in h5_files:
        try:
            info = inspect_magnetic_timeseries_h5(p)
            fmt = str(info.get("attrs", {}).get("format", ""))
            if "timeseries" in fmt.lower():
                ts_cands.append(p)
        except Exception:
            continue
    if not ts_cands:
        raise RuntimeError("No timeseries Mag_*.h5 found.")
    h5_path = max(ts_cands, key=lambda pp: pp.stat().st_mtime)

print("Using:", h5_path)
motion = diagnose_magnetic_h5_mesh_motion(h5_path)
print("mesh_coords_mode:", motion.get("mesh_coords_mode"))
print("moving_node_count:", motion.get("moving_node_count"))
print("moving_reg_codes:", motion.get("moving_reg_codes_labeled") or motion.get("moving_reg_codes"))
print("conclusion:", motion.get("conclusion"))

# 3) Load time series lazily and plot interactively
ts = get_magnetic_timeseries_from_file(h5_path, key="time_index")
print("steps:", len(ts.steps), "range=", (ts.steps[0], ts.steps[-1]))

# Interactive scalar field plot with step slider
interactive_magnetic_plot(ts, quantity=QUANTITY_LOCAL, s=POINT_SIZE_LOCAL, cmap=CMAP_LOCAL)

# Optional: quiver plot (can be heavy). Increase stride if slow.
interactive_magnetic_quiver(ts, stride=2, scale=1, normalize=False)

# Optional: B-locus plot
try:
    interactive_b_locus_field_plot(ts)
except Exception as e:
    print("interactive_b_locus_field_plot skipped:", e)

Found Mag_*.h5: 0


FileNotFoundError: No Mag_*.h5 found. Run the export cell with EXPORT_MAG_H5=True first.

In [15]:
# 3) Debug: Compare TXT vs H5 MagneticRegions (focus: reg_code=94 a2 partial rotation)

from pathlib import Path
import numpy as np

from tools.motorCAD.pyMCAD.magnetic import export_magnetic_txt

REG_CODE = 94  # a2
STEP0 = 0       # index into the time series (0 = first)
STEP1 = -1      # index into the time series (-1 = last)

# Optional: manually set a specific TXT to compare (recommended if there are many .txt)
TXT_PATH_OVERRIDE = None  # e.g. r"E:\\...\\MagTransient.txt"

# Inputs: reuse h5_path from Cell 12 if present
H5_PATH = Path(globals().get("h5_path", "")) if str(globals().get("h5_path", "")).strip() else None
if not H5_PATH or not H5_PATH.exists():
    raise FileNotFoundError("H5 path not found. Run Cell 12 first (it sets variable h5_path).")

# 1) Pick / generate a TXT to compare against
TXT_PATH = None
if TXT_PATH_OVERRIDE:
    TXT_PATH = Path(TXT_PATH_OVERRIDE)
else:
    try:
        cand_dir = H5_PATH.parent
        txt_cands = sorted(list(cand_dir.glob("*.txt")))
        # prioritize likely files
        txt_cands = sorted(
            txt_cands,
            key=lambda p: (
                0 if "mag" in p.name.lower() else 1,
                0 if "trans" in p.name.lower() else 1,
                -p.stat().st_mtime,
            ),
        )
        if txt_cands:
            TXT_PATH = txt_cands[0]
    except Exception:
        TXT_PATH = None

if TXT_PATH is None or not TXT_PATH.exists():
    # Export a compare TXT (uses current Motor-CAD instance `mc`)
    compare_txt = H5_PATH.with_name(H5_PATH.stem + "_compare_export.txt")
    print("No nearby TXT found; exporting:", compare_txt)
    export_magnetic_txt(
        mc,
        first_step=int(globals().get("FIRST_STEP", 1)),
        final_step=int(globals().get("FINAL_STEP", 45)),
        filename=compare_txt,
    )
    TXT_PATH = compare_txt

print("H5:", H5_PATH)
print("TXT:", TXT_PATH)

# 2) Load timeseries from both sources
ts_h5 = get_magnetic_timeseries_from_file(H5_PATH, key="time_index")  # key ignored for H5 adapter
steps_h5 = ts_h5.steps
print("H5 steps[0:5]...:", steps_h5[:5], "len=", len(steps_h5))

# TXT: try to pick a key that matches H5 step keys, otherwise fall back to index-based alignment
def _load_txt_best_effort(path: Path, h5_steps: list[int]):
    ts_a = get_magnetic_timeseries_from_file(path, key="time_index")
    steps_a = ts_a.steps
    if set(steps_a).intersection(set(h5_steps)):
        return ts_a, "time_index"
    ts_b = get_magnetic_timeseries_from_file(path, key="solution")
    steps_b = ts_b.steps
    if set(steps_b).intersection(set(h5_steps)):
        return ts_b, "solution"
    # no match: return time_index but mark mismatch
    return ts_a, "time_index(no_common)"

ts_txt, txt_key_used = _load_txt_best_effort(TXT_PATH, steps_h5)
steps_txt = ts_txt.steps
print("TXT key_used:", txt_key_used)
print("TXT steps[0:5]...:", steps_txt[:5], "len=", len(steps_txt))

N = int(min(len(steps_h5), len(steps_txt)))
if N <= 1:
    raise RuntimeError("Not enough steps to compare. Check that TXT export includes multiple steps.")
steps_h5_use = [int(s) for s in steps_h5[:N]]
steps_txt_use = [int(s) for s in steps_txt[:N]]

common_steps = sorted(set(steps_h5_use).intersection(set(steps_txt_use)))
if not common_steps:
    print("(warn) No common step keys between H5 and TXT.")
    print("       Proceeding with index-based alignment: H5[i] vs TXT[i] for i in [0..N-1].")
else:
    print("(info) Found common step keys; still using index-based range [0..N-1] for fairness.")

def _region_node_ids(mr, reg_code: int) -> set[int]:
    if reg_code <= 0 or reg_code > len(mr):
        return set()
    r = mr[reg_code - 1]
    out: set[int] = set()
    for el in getattr(r, "elements", []) or []:
        out.add(int(el.node_1))
        out.add(int(el.node_2))
        out.add(int(el.node_3))
    return out

def _node_xy(mr):
    return dict(getattr(mr, "node_xy", {}) or {})

def _max_disp_over_steps(ts, node_ids: set[int], steps: list[int]) -> dict[int, float]:
    if not steps:
        return {}
    mr0 = ts.by_step[int(steps[0])]
    xy0 = _node_xy(mr0)
    out: dict[int, float] = {int(n): 0.0 for n in node_ids}
    for st in steps[1:]:
        mr = ts.by_step[int(st)]
        xy = _node_xy(mr)
        for nid in node_ids:
            p0 = xy0.get(int(nid))
            p1 = xy.get(int(nid))
            if p0 is None or p1 is None:
                continue
            try:
                d2 = (float(p1[0]) - float(p0[0])) ** 2 + (float(p1[1]) - float(p0[1])) ** 2
                d = float(d2 ** 0.5)
                if d > out[int(nid)]:
                    out[int(nid)] = d
            except Exception:
                continue
    return out

# Load step0 regions (by index)
step0_h5 = steps_h5_use[int(STEP0)]
step0_txt = steps_txt_use[int(STEP0)]
mr_h5_0 = ts_h5.by_step[int(step0_h5)]
mr_txt_0 = ts_txt.by_step[int(step0_txt)]

nodes_h5 = _region_node_ids(mr_h5_0, REG_CODE)
nodes_txt = _region_node_ids(mr_txt_0, REG_CODE)
print(f"reg_code={REG_CODE}: node_ids in H5(step={step0_h5})={len(nodes_h5)}, in TXT(step={step0_txt})={len(nodes_txt)}")
print("node_ids only in TXT:", len(nodes_txt - nodes_h5))
print("node_ids only in H5:", len(nodes_h5 - nodes_txt))

# Compare motion magnitude per node across the aligned index range
disp_h5 = _max_disp_over_steps(ts_h5, nodes_h5, steps_h5_use)
disp_txt = _max_disp_over_steps(ts_txt, nodes_txt, steps_txt_use)

# Nodes that move in TXT but are nearly static in H5
STATIC_TOL_MM = 1e-6
suspects = []
for nid in sorted(nodes_txt.intersection(nodes_h5)):
    dt = disp_txt.get(int(nid), 0.0)
    dh = disp_h5.get(int(nid), 0.0)
    if dt > 1e-3 and dh <= STATIC_TOL_MM:
        suspects.append((nid, dt, dh))

print("suspects (move in TXT, static in H5):", len(suspects))
for nid, dt, dh in suspects[:20]:
    print(f"  node {nid}: txt_max_disp={dt:.6g} mm, h5_max_disp={dh:.6g} mm")

# H5 internal check: are all reg_code nodes included in moving_node_id?
try:
    import h5py
    with h5py.File(H5_PATH, "r") as f:
        mcm = str(f.attrs.get("mesh_coords_mode", ""))
        if "by_step_moving_nodes" not in mcm.lower():
            print("(info) H5 mesh_coords_mode is not by_step_moving_nodes:", mcm)
        moving_ids = (
            set(int(x) for x in np.asarray(f["mesh/moving_node_id"][:], dtype=np.int32).tolist())
            if "mesh/moving_node_id" in f
            else set()
        )
        rc = np.asarray(f["mesh/reg_code"][:], dtype=np.int32)
        n1 = np.asarray(f["mesh/node_1"][:], dtype=np.int32)
        n2 = np.asarray(f["mesh/node_2"][:], dtype=np.int32)
        n3 = np.asarray(f["mesh/node_3"][:], dtype=np.int32)
        mask = (rc == int(REG_CODE))
        reg_nodes = set(int(x) for x in np.concatenate((n1[mask], n2[mask], n3[mask]), axis=0).tolist())
        missing_in_moving = sorted(reg_nodes - moving_ids)
        print(f"H5 moving_node_id count={len(moving_ids)}")
        print(f"H5 reg_code={REG_CODE} nodes={len(reg_nodes)}")
        print(f"reg_code nodes missing in moving_node_id={len(missing_in_moving)}")
        if missing_in_moving:
            print("  first missing node_ids:", missing_in_moving[:30])
except Exception as e:
    print("(warn) Could not inspect raw H5 moving_node_id:", e)

# Compare a couple of suspect node coordinates at first/last (by index)
step1_h5 = steps_h5_use[int(STEP1)]
step1_txt = steps_txt_use[int(STEP1)]
if suspects:
    nid = suspects[0][0]
    p_txt0 = _node_xy(ts_txt.by_step[int(step0_txt)]).get(int(nid))
    p_txt1 = _node_xy(ts_txt.by_step[int(step1_txt)]).get(int(nid))
    p_h50 = _node_xy(ts_h5.by_step[int(step0_h5)]).get(int(nid))
    p_h51 = _node_xy(ts_h5.by_step[int(step1_h5)]).get(int(nid))
    print("\nExample suspect node:")
    print(f" step0: TXT(step={step0_txt}) {p_txt0} | H5(step={step0_h5}) {p_h50}")
    print(f" step1: TXT(step={step1_txt}) {p_txt1} | H5(step={step1_h5}) {p_h51}")
else:
    print("\nNo suspects found under current thresholds. If you still see partial rotation, it may be: ")
    print("- a2 region actually includes stationary geometry in this result, or")
    print("- the TXT/H5 are not from the exact same case/result (different .mes), or")
    print("- plotting is filtering/skipping elements due to missing node coords.")

FileNotFoundError: H5 path not found. Run Cell 12 first (it sets variable h5_path).